# Problem 2.2 — H₂ SQD Subspace Diagonalization

**Tencent Sparking Program 2026 — Quantum Computing**

This notebook analyzes H₂/STO-3G in two regimes:
1. **Equilibrium** (R=0.74 Å): weak correlation, RHF works well
2. **Stretched** (R=3.0 Å): strong correlation, RHF fails qualitatively

We verify Brillouin's theorem, compute 6×6 CI matrices, and compare RHF vs UHF.

In [1]:
import numpy as np
from openfermion.chem import MolecularData
from openfermionpyscf import run_pyscf
from openfermion import get_fermion_operator, jordan_wigner, get_sparse_operator
from pyscf import gto, scf, fci

def analyze_h2(geometry, label):
    """
    Run full H₂ analysis for a given geometry.
    geometry: list of tuples [(atom, (x,y,z)), ...]
    Returns dict with all results.
    """
    mol_data = MolecularData(geometry, 'sto-3g', multiplicity=1)
    mol_data = run_pyscf(mol_data, run_scf=True, run_fci=True)

    mol_ham = mol_data.get_molecular_hamiltonian()
    ferm_op = get_fermion_operator(mol_ham)
    qubit_op = jordan_wigner(ferm_op)
    H = get_sparse_operator(qubit_op, n_qubits=4)

    # 2-electron configurations
    e2 = sorted([i for i in range(16) if bin(i).count('1') == 2])
    configs = [(v, tuple(sorted(j for j in range(4) if (v >> j) & 1))) for v in e2]

    # Build 6×6 CI matrix
    H6 = np.zeros((6, 6))
    for i, ii in enumerate(e2):
        for j, jj in enumerate(e2):
            H6[i, j] = H[ii, jj].real

    # HF state detection
    hf_idx = np.argmin(np.abs(np.array([H[v, v].real for v in e2]) - mol_data.hf_energy))
    hf_val, hf_occ = configs[hf_idx]

    # Diagonalize 6×6
    eigvals, eigvecs = np.linalg.eigh(H6)

    # HF + singles (5×5): exclude double excitation
    def exc_label(val, occ):
        s = set(occ); r = set(hf_occ) - s; a = s - set(hf_occ)
        n = len(r)
        if n == 0: return 'HF'
        if n == 1: return 'single'
        return 'double'

    singles_idx = [k for k, (val, occ) in enumerate(configs) if exc_label(val, occ) != 'double']
    H5 = H6[np.ix_(singles_idx, singles_idx)]
    ev5, _ = np.linalg.eigh(H5)

    # Gap
    eps = mol_data.orbital_energies
    gap = eps[1] - eps[0]  # LUMO - HOMO

    return {
        'label': label,
        'e_rhf': mol_data.hf_energy,
        'e_fci': mol_data.fci_energy,
        'e_sqd': eigvals[0].real,
        'e_hf_singles': ev5[0].real,
        'eigvals': eigvals,
        'eigvecs': eigvecs,
        'H6': H6,
        'H5': H5,
        'singles_idx': singles_idx,
        'hf_idx': hf_idx,
        'hf_val': hf_val,
        'hf_occ': hf_occ,
        'configs': configs,
        'e2': e2,
        'gap': gap,
        'orb_energies': eps,
        'H_sparse': H,
    }

def print_matrix(mat, row_labels=None, col_labels=None, fmt='>12.8f'):
    """Pretty-print a matrix."""
    n = mat.shape[0]
    if row_labels is None:
        row_labels = [f'D{i}' for i in range(n)]
    if col_labels is None:
        col_labels = [f'D{j}' for j in range(n)]
    header = f"  {'':>5s}" + ''.join(f"{l:>12s}" for l in col_labels)
    print(header)
    print(f"  {'─' * (5 + 12*n)}")
    for i in range(n):
        print(f"  {row_labels[i]:>5s}" + ''.join(f"{mat[i,j]:{fmt}}" for j in range(n)))

print('Imports & helper functions loaded.')
print('Use analyze_h2(geometry, label) to run a full analysis.')

## Part 1 — H₂ at Equilibrium (R = 0.74 Å)

H₂ near its equilibrium bond length. Weakly correlated:
- HOMO-LUMO gap is large (~1.25 Ha)
- HF wavefunction dominates (>98% weight)
- RHF is an excellent approximation

In [ ]:
print('=' * 64)
print('H₂/STO-3G @ R = 0.74 Å (Equilibrium)')
print('=' * 64)

res_eq = analyze_h2(
    [('H', (0, 0, 0)), ('H', (0, 0, 0.74))],
    'Equilibrium (0.74 Å)')

print(f'\nOrbital energies: ε₀ = {res_eq["orb_energies"][0]:+.8f}, '
      f'ε₁ = {res_eq["orb_energies"][1]:+.8f}')
print(f'HOMO-LUMO gap:  {res_eq["gap"]:.4f} Ha')
print(f'E_RHF = {res_eq["e_rhf"]:+.10f} Ha')
print(f'E_FCI = {res_eq["e_fci"]:+.10f} Ha')
print(f'RHF error: {(res_eq["e_rhf"]-res_eq["e_fci"])*27.2114:.4f} eV')

# HF state
print(f'\n--- HF State ---')
print(f'|HF⟩ = |{res_eq["hf_val"]:04b}⟩  (spin orbitals {res_eq["hf_occ"]} occupied)')
print(f'⟨HF|H|HF⟩ = {res_eq["e_rhf"]:+.10f} Ha = E_RHF ✓')
print(f'Preparation: X on qubits {res_eq["hf_occ"]}')

# 6 configurations
print(f'\n--- 6 Two-electron Configurations ---')
print(f'  {"k":>3}  {"int":>4}  {"|b₃b₂b₁b₀⟩":>12}  {"occ":>10}  {"type":>10}')
for k, (val, occ) in enumerate(res_eq['configs']):
    s = set(occ); r = set(res_eq['hf_occ']) - s
    n = len(r); t = 'HF' if n==0 else ('single' if n==1 else 'double')
    print(f'  {k:>3}  {val:>4}  {"|" + f"{val:04b}" + "⟩":>12}  {str(occ):>10}  {t:>10}')

# 6×6 CI matrix
print(f'\n--- 6×6 CI Matrix (Ha) ---')
print_matrix(res_eq['H6'])

# Eigenvalues
print(f'\n6×6 eigenvalues:')
for k, ev in enumerate(res_eq['eigvals']):
    tag = ' ← ground' if k == 0 else ''
    print(f'  λ{k} = {ev.real:+.10f} Ha{tag}')

print(f'\nE_SQD = E_FCI = {res_eq["e_sqd"]:+.10f} Ha  (Δ = {abs(res_eq["e_sqd"]-res_eq["e_fci"]):.2e})')

# Ground-state composition
gs = res_eq['eigvecs'][:, 0]
print(f'\nGround state |Ψ₀⟩ composition:')
contrib = sorted([(abs(c)**2, k, c) for k, c in enumerate(gs)], reverse=True)
for w, k, c in contrib:
    val, occ = res_eq['configs'][k]
    t = 'HF' if res_eq['hf_idx']==k else ('single' if len(set(res_eq['hf_occ'])-set(occ))==1 else 'double')
    print(f'  |D{k}⟩ = |{val:04b}⟩  coeff={c.real:+9.6f}  |c|²={w:.5f}  ({t})')

# Brillouin: HF + singles
print(f'\n--- HF + Singles (5×5) — Brillouin Verification ---')
print(f'\n5×5 CI matrix:')
sl = res_eq['singles_idx']
print_matrix(res_eq['H5'], [f'D{s}' for s in sl], [f'D{s}' for s in sl])

print(f'\n⟨HF|H|single⟩ matrix elements:')
for j, sj in enumerate(sl):
    if sj != res_eq['hf_idx']:
        val, occ = res_eq['configs'][sj]
        print(f'  ⟨HF|H|D{sj}(|{val:04b}⟩)⟩ = {res_eq["H6"][res_eq["hf_idx"],sj]:+.8e}')

brill_max = max(abs(res_eq['H6'][res_eq['hf_idx'], sj]) for sj in sl if sj != res_eq['hf_idx'])
print(f'\nmax |⟨HF|H|single⟩| = {brill_max:.2e}  →  Brillouin satisfied ✓')
print(f'E(HF+singles) = {res_eq["e_hf_singles"]:+.10f} Ha')
print(f'E_RHF          = {res_eq["e_rhf"]:+.10f} Ha')
print(f'Δ = {abs(res_eq["e_hf_singles"]-res_eq["e_rhf"]):.2e}  →  equal ✓')

## Part 2 — H₂ at Stretched Bond (R = 3.0 Å)

H₂ stretched to 3.0 Å (nearly dissociated). **Strongly correlated**:
- HOMO-LUMO gap nearly vanishes (~0.2 Ha)
- HF wavefunction is ~54% weight, double excitation is ~46%
- RHF gives qualitatively wrong energy (~7.6 eV error)
- But Brillouin's theorem still holds **mathematically**

In [ ]:
print('=' * 64)
print('H₂/STO-3G @ R = 3.0 Å (Stretched / Near-dissociated)')
print('=' * 64)

res_st = analyze_h2(
    [('H', (0, 0, 0)), ('H', (0, 0, 3.0))],
    'Stretched (3.0 Å)')

print(f'\nOrbital energies: ε₀ = {res_st["orb_energies"][0]:+.8f}, '
      f'ε₁ = {res_st["orb_energies"][1]:+.8f}')
print(f'HOMO-LUMO gap:  {res_st["gap"]:.4f} Ha  ← nearly degenerate!')
print(f'E_RHF = {res_st["e_rhf"]:+.10f} Ha')
print(f'E_FCI = {res_st["e_fci"]:+.10f} Ha')
print(f'RHF error: {(res_st["e_rhf"]-res_st["e_fci"])*27.2114:.4f} eV  ← ~7.5 eV! RHF fails!')

# HF state
print(f'\n--- HF State ---')
print(f'|HF⟩ = |{res_st["hf_val"]:04b}⟩  (spin orbitals {res_st["hf_occ"]} occupied)')
print(f'⟨HF|H|HF⟩ = {res_st["e_rhf"]:+.10f} Ha')

# 6×6 CI matrix
print(f'\n--- 6×6 CI Matrix (Ha) — note HF-DOUBLE coupling is large ---')
print_matrix(res_st['H6'])

# Eigenvalues
print(f'\n6×6 eigenvalues:')
for k, ev in enumerate(res_st['eigvals']):
    tag = ' ← ground' if k == 0 else ''
    print(f'  λ{k} = {ev.real:+.10f} Ha{tag}')

print(f'\nE_SQD = E_FCI = {res_st["e_sqd"]:+.10f} Ha')

# Ground-state composition — double excitation is significant!
gs = res_st['eigvecs'][:, 0]
print(f'\nGround state |Ψ₀⟩ composition:')
contrib = sorted([(abs(c)**2, k, c) for k, c in enumerate(gs)], reverse=True)
for w, k, c in contrib:
    val, occ = res_st['configs'][k]
    t = 'HF' if res_st['hf_idx']==k else ('single' if len(set(res_st['hf_occ'])-set(occ))==1 else 'double')
    print(f'  |D{k}⟩ = |{val:04b}⟩  coeff={c.real:+9.6f}  |c|²={w:.5f}  ({t})')

# Brillouin: still holds at 3.0 Å!
print(f'\n--- HF + Singles (5×5) — Brillouin Still Holds! ---')
sl = res_st['singles_idx']
print_matrix(res_st['H5'], [f'D{s}' for s in sl], [f'D{s}' for s in sl])

print(f'\n⟨HF|H|single⟩ matrix elements:')
for j, sj in enumerate(sl):
    if sj != res_st['hf_idx']:
        val, occ = res_st['configs'][sj]
        print(f'  ⟨HF|H|D{sj}(|{val:04b}⟩)⟩ = {res_st["H6"][res_st["hf_idx"],sj]:+.8e}')

brill_st = max(abs(res_st['H6'][res_st['hf_idx'], sj]) for sj in sl if sj != res_st['hf_idx'])
print(f'\nmax |⟨HF|H|single⟩| = {brill_st:.2e}  →  Brillouin STILL satisfied ✓')
print(f'Even at 3.0 Å, ⟨HF|H|single⟩ = 0 exactly!')

print(f'\n5×5 eigenvalues:')
for k, ev in enumerate(res_st['eigvals']):
    if k == 0:
        continue
    eig_tag = '' if k < 5 else ''
    print(f'  λ{k} = {ev.real:+.10f} Ha{eig_tag}')

# Show that lowest 5×5 eigenvalue comes from singles, not HF
print(f'\nE(HF+singles) = {res_st["e_hf_singles"]:+.10f} Ha  (lowest of 5×5)')
print(f'E_RHF          = {res_st["e_rhf"]:+.10f} Ha')
print(f'Δ = {abs(res_st["e_hf_singles"]-res_st["e_rhf"]):.4f} Ha ≠ 0  ← singles block dips BELOW E_RHF!')
print(f'\nBecause ⟨HF|H|single⟩ = 0, the 5×5 matrix is block-diagonal:')
print(f'  eigenvalues = {{E_HF, evals(H_singles_4×4)}}')
print(f'At 3.0 Å, the singles block has an eigenvalue ({res_st["e_hf_singles"]:+.4f})')
print(f'below E_RHF ({res_st["e_rhf"]:+.4f}).')
print(f'{chr(8594)} This does NOT violate Brillouin. It shows that the singles')
print(f'    block eigenstates (spin-flip excitations) are energetically favored.')
print(f'{chr(8594)} But only including doubles gives E_FCI = {res_st["e_fci"]:+.4f} Ha')

## Why RHF Fails: RHF vs FCI Gap Analysis

| Quantity | 0.74 Å | 3.0 Å |
|:---|:---:|:---:|
| ε_HOMO | -0.58 Ha | -0.18 Ha |
| ε_LUMO | +0.67 Ha | +0.02 Ha |
| **Δε (gap)** | **~1.25 Ha** | **~0.20 Ha** |
| HF weight in Ψ₀ | ~99% | ~70% |
| Double weight | ~1% | ~30% |
| RHF error | ~0.02 Ha (~0.6 eV) | ~0.28 Ha (~7.6 eV) |

Key insight: When the gap vanishes, σ_g and σ_u become nearly degenerate.
The true wavefunction becomes an equal-weight superposition:
$$|\Psi_0\rangle \approx \frac{1}{\sqrt{2}}(|\sigma_g\bar{\sigma}_g\rangle - |\sigma_u\bar{\sigma}_u\rangle)$$
RHF can only represent one Slater determinant — it's **qualitatively wrong**.

In [ ]:
# Direct comparison of the two regimes
print('Comparison: Equilibrium vs Stretched')
print('=' * 60)
print(f'{"":>20s}  {"0.74 Å":>15s}  {"3.0 Å":>15s}')
print(f'{"─"*60}')
print(f'{"HOMO-LUMO gap":>20s}  {res_eq["gap"]:>15.4f}  {res_st["gap"]:>15.4f} Ha')
print(f'{"E_RHF":>20s}  {res_eq["e_rhf"]:>15.8f}  {res_st["e_rhf"]:>15.8f} Ha')
print(f'{"E_FCI":>20s}  {res_eq["e_fci"]:>15.8f}  {res_st["e_fci"]:>15.8f} Ha')
print(f'{"RHF error":>20s}  {abs(res_eq["e_rhf"]-res_eq["e_fci"])*27.2114:>15.4f}  '
      f'{abs(res_st["e_rhf"]-res_st["e_fci"])*27.2114:>15.4f} eV')
print(f'{"E(HF+singles)":>20s}  {res_eq["e_hf_singles"]:>15.8f}  {res_st["e_hf_singles"]:>15.8f} Ha')
eq_check = '✓' if abs(res_eq["e_hf_singles"] - res_eq["e_rhf"]) < 1e-6 else '✗'
st_check = '✓' if abs(res_st["e_hf_singles"] - res_st["e_rhf"]) < 1e-6 else '✗'
print(f'{"= E_RHF?":>20s}  {eq_check:>15s}  {st_check:>15s}')
print(f'{"Brillouin holds?":>20s}  {"✓":>15s}  {"✓":>15s}')

# HF weight comparison
hf_w_eq = abs(res_eq['eigvecs'][res_eq['hf_idx'], 0])**2
hf_w_st = abs(res_st['eigvecs'][res_st['hf_idx'], 0])**2
print(f'{"HF weight in Ψ₀":>20s}  {hf_w_eq:>15.4f}  {hf_w_st:>15.4f}')

# Find double excitation weight
for k, (val, occ) in enumerate(res_eq['configs']):
    if set(res_eq['hf_occ']) != set(occ) and len(set(res_eq['hf_occ'])-set(occ)) >= 2:
        d_eq = abs(res_eq['eigvecs'][k, 0])**2
        break
for k, (val, occ) in enumerate(res_st['configs']):
    if set(res_st['hf_occ']) != set(occ) and len(set(res_st['hf_occ'])-set(occ)) >= 2:
        d_st = abs(res_st['eigvecs'][k, 0])**2
        break
print(f'{"Double weight in Ψ₀":>20s}  {d_eq:>15.4f}  {d_st:>15.4f}')
print(f'\n→ At 3.0 Å, the double excitation is ~{d_st*100:.0f}% — NOT a small correction!')

## Part 3 — RHF vs UHF: Breaking Spin Symmetry

UHF allows α and β electrons to have **different spatial orbitals**, achieving the correct
dissociation limit (one electron on each H atom). This requires a **broken-symmetry initial guess**.

The price: spin contamination — ⟨Ŝ²⟩ ≠ 0 for a nominal singlet.

For the initial guess, we start from the RHF solution and mix HOMO (σ_g) and LUMO (σ_u)
with **opposite signs** for α and β, creating spatial separation.

In [ ]:
print('=' * 64)
print('RHF vs UHF vs FCI at Equilibrium & Stretched Bond')
print('=' * 64)

def run_uhf(bond_length, label):
    """Run UHF with broken-symmetry initial guess."""
    mol = gto.M(atom=f'H 0 0 0; H 0 0 {bond_length}',
                basis='sto-3g', spin=0, verbose=0)

    # SCF options for difficult convergence
    mf_rhf = scf.RHF(mol)
    mf_rhf.kernel()

    # Build broken-symmetry guess by mixing HOMO-LUMO with opposite signs
    theta = 0.3
    mo = mf_rhf.mo_coeff
    mo_a = mo.copy()
    mo_b = mo.copy()
    c, s = np.cos(theta), np.sin(theta)
    # α: rotate by +θ (more weight on atom A)
    mo_a[:, [0, 1]] = mo_a[:, [0, 1]] @ np.array([[c, -s], [s, c]])
    # β: rotate by -θ (more weight on atom B)
    mo_b[:, [0, 1]] = mo_b[:, [0, 1]] @ np.array([[c, s], [-s, c]])

    dm_a = mo_a[:, [0]] @ mo_a[:, [0]].T
    dm_b = mo_b[:, [0]] @ mo_b[:, [0]].T

    mf_uhf = scf.UHF(mol)
    mf_uhf.kernel(dm0=(dm_a, dm_b))

    s2, _ = mf_uhf.spin_square()
    return mf_uhf.e_tot, mf_rhf.e_tot, s2

# Run at both geometries
e_uhf_eq, e_rhf_eq, s2_eq = run_uhf(0.74, 'equilibrium')
e_uhf_st, e_rhf_st, s2_st = run_uhf(3.0, 'stretched')

print(f'\n{"":>20s}  {"RHF":>12s}  {"UHF":>12s}  {"FCI":>12s}  {"⟨S²⟩(UHF)":>10s}')
print(f'{"─" * 72}')
print(f'{"Equilibrium (0.74 Å)":>20s}  {res_eq["e_rhf"]:>12.8f}  '
      f'{e_uhf_eq:>12.8f}  {res_eq["e_fci"]:>12.8f}  {s2_eq:>10.4f}')
print(f'{"Stretched  (3.0 Å)":>20s}  {res_st["e_rhf"]:>12.8f}  '
      f'{e_uhf_st:>12.8f}  {res_st["e_fci"]:>12.8f}  {s2_st:>10.4f}')

# Errors
print(f'\n{"":>20s}  {"RHF err (eV)":>12s}  {"UHF err (eV)":>12s}')
print(f'{"─" * 48}')
print(f'{"Equilibrium":>20s}  '
      f'{abs(res_eq["e_rhf"]-res_eq["e_fci"])*27.2114:>12.4f}  '
      f'{abs(e_uhf_eq-res_eq["e_fci"])*27.2114:>12.4f}')
print(f'{"Stretched":>20s}  '
      f'{abs(res_st["e_rhf"]-res_st["e_fci"])*27.2114:>12.4f}  '
      f'{abs(e_uhf_st-res_st["e_fci"])*27.2114:>12.4f}')

print(f'\n--- UHF Analysis ---')
print(f'At equilibrium: UHF ≈ RHF (no symmetry breaking needed), ⟨S²⟩ ≈ 0')
print(f'At 3.0 Å:       UHF ≫ RHF (much closer to FCI!),   ⟨S²⟩ ≈ 1.0')
print(f'{chr(8594)} UHF breaks spin symmetry but captures the correct physics')
print(f'{chr(8594)} ⟨S²⟩ = {s2_st:.1f} means ~50% singlet + ~50% triplet contamination')

## Summary

### Brillouin's Theorem

$$\langle\Phi_0|\hat{H}|\Phi_i^a\rangle = f_{ia} = 0$$

This is a **mathematical identity** from the HF equations — it holds at any bond length.
Consequence: the 5×5 CI matrix (HF + singles) is **block diagonal**, with eigenvalues
= {E_RHF} ∪ {evals(H_singles block)}. At equilibrium, E_RHF < all singles eigenvalues,
so E(HF+singles) = E_RHF. At 3.0 Å, singles block eigenvalues can dip below E_RHF.

### When Brillouin Is Not Enough

| Regime | Gap | HF weight | Singles help? | Doubles help? |
|:---|:---:|:---:|:---:|:---:|
| Weak correlation (0.74 Å) | Large | ~99% | No (Brillouin) | Small correction |
| Strong correlation (3.0 Å) | → 0 | ~54% | **Singles block < E_HF!** | **Essential (46%)** |

### RHF vs UHF at 3.0 Å

- **RHF**: wrong dissociation (50% H⁻H⁺, 50% H·H·), ~7.6 eV error
- **UHF**: correct dissociation (H· + H·), ~0.01 eV error
- **Cost**: spin contamination (⟨Ŝ²⟩ ≈ 1.0 instead of 0)

The **strongly correlated regime** is defined by near-degeneracy of orbitals,
not by shorter inter-electronic distances. When multiple Slater determinants
are nearly degenerate, no single determinant can describe the system —
multi-reference methods (CASSCF, CASCI) are required.